# NVIDIA Nemotron Model Reasoning Challenge

Minimal Kaggle notebook for a first valid LoRA submission.

- Reads `train.csv` and `test.csv`
- Builds supervised fine-tuning JSONL files
- Fine-tunes a LoRA adapter on the attached Nemotron base model
- Saves the adapter and packages `submission.zip`


In [ ]:
# ============================================================
# 0. Prepare runtime dependencies
# ============================================================
import os
import re as _re
import shutil
import subprocess
import sys
import zipfile
import ctypes
import glob as glob_mod
import importlib.metadata as importlib_metadata
from importlib import import_module

ALLOW_ONLINE_INSTALL = False
UTILITY_SCRIPT_DIRS = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script",
    "/kaggle/usr/lib/nvidia-utility-script",
]
RUNTIME_SITE_DIR = "/kaggle/working/local_site_packages" if os.path.isdir("/kaggle/working") else "/tmp/local_site_packages"
TRITON_BIN_DIR = "/kaggle/working/triton-bin" if os.path.isdir("/kaggle/working") else "/tmp/triton-bin"
TRITON_CACHE_DIR = "/kaggle/working/triton-cache" if os.path.isdir("/kaggle/working") else "/tmp/triton-cache"

# --- Locate local wheel directories ---
WHEEL_DIR = None
for candidate in [
    "/kaggle/input/nemotron-deps-wheels",
    "/kaggle/input/nemotron-deps-wheels/nemotron-deps-wheels",
]:
    if os.path.isdir(candidate):
        WHEEL_DIR = candidate
        break

PYTORCH_DIR = None
for candidate in [
    "/kaggle/input/pytorch-nightly-cu128",
    "/kaggle/input/pytorch-nightly-cu128/pytorch-nightly-cu128",
]:
    if os.path.isdir(candidate):
        PYTORCH_DIR = candidate
        break

def _activate_runtime_site_dir(reset=False):
    if reset and os.path.isdir(RUNTIME_SITE_DIR):
        shutil.rmtree(RUNTIME_SITE_DIR)
    os.makedirs(RUNTIME_SITE_DIR, exist_ok=True)
    if RUNTIME_SITE_DIR in sys.path:
        sys.path.remove(RUNTIME_SITE_DIR)
    sys.path.insert(0, RUNTIME_SITE_DIR)
    existing = [part for part in os.environ.get("PYTHONPATH", "").split(os.pathsep) if part]
    if RUNTIME_SITE_DIR in existing:
        existing.remove(RUNTIME_SITE_DIR)
    os.environ["PYTHONPATH"] = os.pathsep.join([RUNTIME_SITE_DIR] + existing)
    return RUNTIME_SITE_DIR

_activate_runtime_site_dir(reset=True)

def _activate_utility_script_paths():
    """Prepend package directories from Kaggle utility scripts if present."""
    roots = []
    for path in UTILITY_SCRIPT_DIRS:
        if os.path.isdir(path) and path not in roots:
            roots.append(path)
    roots.extend(
        sorted(
            path for path in glob_mod.glob("/kaggle/usr/lib/notebooks/*/nvidia-utility-script")
            if os.path.isdir(path) and path not in roots
        )
    )

    added = []
    seen = set()
    for root in roots:
        candidate_dirs = [root]
        for current, dirs, _ in os.walk(root):
            depth = os.path.relpath(current, root).count(os.sep)
            if depth >= 3:
                dirs[:] = []
            if os.path.isfile(os.path.join(current, "torch", "__init__.py")) or os.path.isfile(
                os.path.join(current, "mamba_ssm", "__init__.py")
            ):
                candidate_dirs.append(current)

        for path in candidate_dirs:
            if path in seen:
                continue
            seen.add(path)
            if path not in sys.path:
                sys.path.insert(0, path)
                added.append(path)

    if added:
        print("Attached utility script package paths:")
        for path in added:
            print(f"  {path}")
    return added

UTILITY_PACKAGE_DIRS = _activate_utility_script_paths()

# Remove utility script sub-paths that shadow real packages (torch, etc.)
_shadow_packages = {"torch", "tilelang"}
_paths_to_remove = []
for p in list(sys.path):
    if "/nvidia-utility-script" not in p:
        continue
    # Keep the root utility script path, remove sub-paths that shadow packages
    for pkg in _shadow_packages:
        if os.path.isdir(os.path.join(p, pkg)):
            # This is a sub-path (not root) that contains a shadowing package
            if p.count("/") > 5:  # sub-directory, not root
                _paths_to_remove.append(p)
                break
for p in _paths_to_remove:
    if p in sys.path:
        sys.path.remove(p)
        print(f"  Removed shadowing path: {p}")

def _copy_executable_tool(src_path):
    if not src_path or not os.path.isfile(src_path):
        return None
    os.makedirs(TRITON_BIN_DIR, exist_ok=True)
    dst_path = os.path.join(TRITON_BIN_DIR, os.path.basename(src_path))
    if not os.path.exists(dst_path) or os.path.getsize(dst_path) != os.path.getsize(src_path):
        shutil.copy2(src_path, dst_path)
    os.chmod(dst_path, 0o755)
    return dst_path

def _prepare_triton_toolchain():
    os.makedirs(TRITON_CACHE_DIR, exist_ok=True)
    os.environ.setdefault("TRITON_CACHE_DIR", TRITON_CACHE_DIR)
    os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", TRITON_CACHE_DIR)

    bin_dirs = []
    seen = set()
    for root in UTILITY_SCRIPT_DIRS + UTILITY_PACKAGE_DIRS:
        if not root or not os.path.isdir(root):
            continue
        bin_dir = os.path.join(root, "triton", "backends", "nvidia", "bin")
        if os.path.isdir(bin_dir) and bin_dir not in seen:
            seen.add(bin_dir)
            bin_dirs.append(bin_dir)

    prepared = {}
    env_map = {
        "ptxas": ("TRITON_PTXAS_PATH",),
        "ptxas-blackwell": ("TRITON_PTXAS_BLACKWELL_PATH", "TRITON_PTXAS-BLACKWELL_PATH"),
        "cuobjdump": ("TRITON_CUOBJDUMP_PATH",),
        "nvdisasm": ("TRITON_NVDISASM_PATH",),
    }
    for binary, env_keys in env_map.items():
        for bin_dir in bin_dirs:
            src_path = os.path.join(bin_dir, binary)
            if not os.path.isfile(src_path):
                continue
            dst_path = _copy_executable_tool(src_path)
            if not dst_path:
                continue
            for env_key in env_keys:
                os.environ[env_key] = dst_path
            prepared[binary] = dst_path
            break

    path_parts = [part for part in os.environ.get("PATH", "").split(os.pathsep) if part]
    if TRITON_BIN_DIR not in path_parts:
        os.environ["PATH"] = os.pathsep.join([TRITON_BIN_DIR] + path_parts)

    if prepared:
        print("Prepared local Triton toolchain:")
        for binary, path in prepared.items():
            print(f"  {binary}: {path}")

_prepare_triton_toolchain()

def _fix_wheel_filename(src_path):
    """Kaggle strips '+' from wheel filenames. Copy to /tmp with '+' restored."""
    basename = os.path.basename(src_path)
    fixed = _re.sub(r"(\d)(cu\d+|git[0-9a-f]+)", r"\1+\2", basename)
    if fixed == basename:
        return src_path
    dst = os.path.join("/tmp", fixed)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src_path):
        shutil.copy2(src_path, dst)
    print(f"  Fixed filename: {basename} -> {fixed}")
    return dst

def _find_wheels(directory, pattern):
    if not directory:
        return []
    matches = sorted(glob_mod.glob(os.path.join(directory, pattern)))
    if matches:
        return matches
    pattern_no_plus = pattern.replace("+", "")
    return sorted(glob_mod.glob(os.path.join(directory, pattern_no_plus)))

def _find_wheel(directory, pattern):
    matches = _find_wheels(directory, pattern)
    return matches[0] if matches else None

def _iter_all_wheel_dirs():
    seen = set()
    for directory in [WHEEL_DIR, PYTORCH_DIR]:
        if directory and os.path.isdir(directory) and directory not in seen:
            seen.add(directory)
            yield directory
    kaggle_input = "/kaggle/input"
    if not os.path.isdir(kaggle_input):
        return
    for path in sorted(glob_mod.glob(os.path.join(kaggle_input, "*"))):
        if os.path.isdir(path):
            if glob_mod.glob(os.path.join(path, "*.whl")) and path not in seen:
                seen.add(path)
                yield path
        nested = os.path.join(path, os.path.basename(path))
        if os.path.isdir(nested) and glob_mod.glob(os.path.join(nested, "*.whl")) and nested not in seen:
            seen.add(nested)
            yield nested

def _wheel_seems_compatible(whl_path):
    basename = os.path.basename(whl_path)
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    return any(
        marker in basename
        for marker in [
            f"-{py_tag}-",
            "-abi3-",
            "-py3-none-",
            "-py2.py3-",
        ]
    )

def _iter_runtime_lib_dirs():
    patterns = [
        os.path.join(RUNTIME_SITE_DIR, "nvidia", "*", "lib"),
        os.path.join(RUNTIME_SITE_DIR, "*", "lib"),
    ]
    seen = set()
    for pattern in patterns:
        for path in sorted(glob_mod.glob(pattern)):
            if os.path.isdir(path) and path not in seen:
                seen.add(path)
                yield path

def _preload_local_cuda_runtime_libs():
    """Force-load local CUDA/NCCL libs so torch does not bind to older system copies."""
    lib_patterns = [
        "libcublasLt.so.*[0-9]",
        "libcublas.so.*[0-9]",
        "libcudnn.so.*[0-9]",
        "libnvrtc.so.*[0-9]",
        "libnvrtc-builtins.so.*[0-9]",
        "libcudart.so.*[0-9]",
        "libcupti.so.*[0-9]",
        "libcufft.so.*[0-9]",
        "libcurand.so.*[0-9]",
        "libnvJitLink.so.*[0-9]",
        "libcusparse.so.*[0-9]",
        "libcusparseLt.so.*[0-9]",
        "libcusolver.so.*[0-9]",
        "libnccl.so.*[0-9]",
        "libnvshmem_host.so.*[0-9]",
        "libcufile.so.*[0-9]",
        "libnvToolsExt.so.*[0-9]",
    ]
    loaded = []
    for pattern in lib_patterns:
        lib_path = None
        for directory in _iter_runtime_lib_dirs():
            matches = sorted(glob_mod.glob(os.path.join(directory, pattern)))
            if matches:
                lib_path = matches[0]
                break
        if not lib_path:
            continue
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
            loaded.append(os.path.basename(lib_path))
        except OSError as exc:
            print(f"  WARNING: failed to preload {os.path.basename(lib_path)}: {exc}")
    if loaded:
        print("Preloaded local CUDA runtime libraries:")
        for name in loaded:
            print(f"  {name}")

def _pip_install_local(whl_path, force_reinstall=False, no_deps=False):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--target",
        RUNTIME_SITE_DIR,
        "--upgrade",
        "--ignore-installed",
    ]
    for link_dir in list(_iter_all_wheel_dirs()) + ["/tmp"]:
        if os.path.isdir(link_dir):
            cmd.extend(["--find-links", link_dir])
    if force_reinstall:
        cmd.append("--force-reinstall")
    if no_deps:
        cmd.append("--no-deps")
    cmd.append(whl_path)
    subprocess.check_call(cmd)
    _activate_runtime_site_dir(reset=False)

def _raise_offline_dependency_error(module_name, wheel_pattern=None, extra_message=None):
    detail = f" Missing wheel pattern: {wheel_pattern}." if wheel_pattern else ""
    extra = f" {extra_message}" if extra_message else ""
    raise FileNotFoundError(
        f"{module_name} is required, but no usable local wheel was found.{detail}"
        f" Attach the Kaggle dataset 'masanorisuda/nemotron-deps-wheels' or add a matching wheel.{extra}"
    )

def _normalize_dist_name(name):
    return _re.sub(r"[-_.]+", "-", name).lower()

_INVALID_WHEEL_WARNED = set()

def _read_wheel_metadata(whl_path):
    try:
        with zipfile.ZipFile(whl_path) as zf:
            metadata_name = None
            for name in zf.namelist():
                if name.endswith("METADATA") and ".dist-info/" in name:
                    metadata_name = name
                    break
            if not metadata_name:
                return None, None, []
            data = zf.read(metadata_name).decode("utf-8", "replace")
    except (OSError, zipfile.BadZipFile, KeyError) as exc:
        basename = os.path.basename(whl_path)
        if basename not in _INVALID_WHEEL_WARNED:
            print(f"  Skipping invalid wheel file: {basename} ({exc})")
            _INVALID_WHEEL_WARNED.add(basename)
        return None, None, []

    dist_name = None
    version = None
    requires = []
    for line in data.splitlines():
        if line.startswith("Name: "):
            dist_name = line.split(": ", 1)[1].strip()
        elif line.startswith("Version: "):
            version = line.split(": ", 1)[1].strip()
        elif line.startswith("Requires-Dist: "):
            requires.append(line.split(": ", 1)[1].strip())
    return dist_name, version, requires

def _find_dist_wheel(dist_names):
    normalized_targets = {_normalize_dist_name(name) for name in dist_names}
    for directory in _iter_all_wheel_dirs():
        for whl_path in sorted(glob_mod.glob(os.path.join(directory, "*.whl"))):
            if not _wheel_seems_compatible(whl_path):
                continue
            dist_name, _, _ = _read_wheel_metadata(whl_path)
            if dist_name and _normalize_dist_name(dist_name) in normalized_targets:
                return whl_path
    return None

def _install_local_torch_runtime_deps():
    """Install extra binary deps required by the Blackwell torch nightly if present locally."""
    dep_map = {
        "nvidia-cublas-cu12": ["nvidia-cublas-cu12"],
        "nvidia-cuda-cupti-cu12": ["nvidia-cuda-cupti-cu12"],
        "nvidia-cuda-nvrtc-cu12": ["nvidia-cuda-nvrtc-cu12"],
        "nvidia-cuda-runtime-cu12": ["nvidia-cuda-runtime-cu12"],
        "nvidia-cufft-cu12": ["nvidia-cufft-cu12"],
        "nvidia-cufile-cu12": ["nvidia-cufile-cu12"],
        "nvidia-curand-cu12": ["nvidia-curand-cu12"],
        "nvidia-cusolver-cu12": ["nvidia-cusolver-cu12"],
        "nvidia-cusparse-cu12": ["nvidia-cusparse-cu12"],
        "nvidia-nvjitlink-cu12": ["nvidia-nvjitlink-cu12"],
        "nvidia-nvtx-cu12": ["nvidia-nvtx-cu12"],
        "cuda-bindings": ["cuda-bindings"],
        "nvidia-cudnn-cu12": ["nvidia-cudnn-cu12"],
        "nvidia-cusparselt-cu12": ["nvidia-cusparselt-cu12"],
        "nvidia-nccl-cu12": ["nvidia-nccl-cu12"],
        "nvidia-nvshmem-cu12": ["nvidia-nvshmem-cu12"],
        "triton": ["triton", "pytorch-triton"],
    }

    found = {}
    missing = []
    for display_name, dist_names in dep_map.items():
        whl = _find_dist_wheel(dist_names)
        if whl:
            found[display_name] = _fix_wheel_filename(whl)
        else:
            missing.append(display_name)

    if found:
        print("  Installing local torch runtime dependency wheels:")
        for display_name, whl in found.items():
            print(f"    {display_name}: {os.path.basename(whl)}")
            _pip_install_local(whl, force_reinstall=True)

    if missing:
        print(f"  Missing optional local torch runtime wheels: {missing}")
    return missing

def _query_gpu_info():
    """Query GPU name and compute capability without importing torch."""
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,compute_cap",
                "--format=csv,noheader",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        return None, None

    if not out:
        return None, None

    first = out.splitlines()[0]
    parts = [part.strip() for part in first.split(",")]
    if len(parts) != 2:
        return first, None
    return parts[0], parts[1]

def _get_installed_torch_dist():
    try:
        dist = importlib_metadata.distribution("torch")
    except importlib_metadata.PackageNotFoundError:
        return None, None
    return dist.version, str(dist.locate_file(""))

# --- Step 0: Upgrade PyTorch if GPU needs it (Blackwell sm_120) ---
def _maybe_upgrade_pytorch():
    """Install local PyTorch nightly with CUDA 12.8 for Blackwell GPUs."""
    gpu_name, compute_cap = _query_gpu_info()
    if gpu_name:
        print(f"GPU: {gpu_name}, compute capability: sm_{compute_cap.replace('.', '') if compute_cap else 'unknown'}")
    if not compute_cap:
        return

    try:
        cap_major = int(compute_cap.split(".")[0])
    except Exception:
        return

    if cap_major <= 9:
        print("PyTorch supports this GPU, no upgrade needed.")
        return

    installed_torch_version, installed_torch_root = _get_installed_torch_dist()
    if installed_torch_version and "dev" in installed_torch_version and any(
        cuda_tag in installed_torch_version for cuda_tag in ["cu128", "cu130"]
    ):
        print("Blackwell GPU detected — using already-installed CUDA nightly torch:")
        print(f"  version: {installed_torch_version}")
        if installed_torch_root:
            print(f"  location: {installed_torch_root}")
        return

    utility_torch_paths = [
        path for path in UTILITY_PACKAGE_DIRS if os.path.isfile(os.path.join(path, "torch", "__init__.py"))
    ]
    if utility_torch_paths:
        print("Blackwell GPU detected — using attached utility script runtime:")
        for path in utility_torch_paths:
            print(f"  {path}")
        return

    if not PYTORCH_DIR:
        raise FileNotFoundError(
            "Blackwell GPU detected, but the Kaggle dataset 'masanorisuda/pytorch-nightly-cu128' "
            "is not attached."
        )

    print(f"Blackwell GPU detected — upgrading PyTorch from {PYTORCH_DIR}")
    whl_files = sorted(glob_mod.glob(os.path.join(PYTORCH_DIR, "*.whl")))
    print(f"  Found wheels: {[os.path.basename(w) for w in whl_files]}")

    triton_whl = _find_wheel(PYTORCH_DIR, "pytorch_triton-*.whl")
    torch_whl = _find_wheel(PYTORCH_DIR, "torch-*.whl")
    if not triton_whl or not torch_whl:
        raise FileNotFoundError(
            "Missing local PyTorch nightly wheels. Expected both 'pytorch_triton-*.whl' and 'torch-*.whl'."
        )

    triton_whl = _fix_wheel_filename(triton_whl)
    torch_whl = _fix_wheel_filename(torch_whl)

    missing_runtime_wheels = _install_local_torch_runtime_deps()

    print(f"  Installing {os.path.basename(triton_whl)}")
    _pip_install_local(triton_whl, force_reinstall=True, no_deps=True)
    print(f"  Installing {os.path.basename(torch_whl)}")
    _pip_install_local(torch_whl, force_reinstall=True, no_deps=True)

    if missing_runtime_wheels:
        print("  Torch nightly installed, but some runtime dependency wheels were not found locally.")

_maybe_upgrade_pytorch()
_preload_local_cuda_runtime_libs()

try:
    import torch
except ImportError as exc:
    if "ncclCommResume" in str(exc):
        raise RuntimeError(
            "PyTorch nightly was installed, but the required CUDA/NCCL runtime wheels are missing or too old "
            "for this Blackwell build. The current torch wheel metadata requires at least "
            "`nvidia-nccl-cu12==2.29.7`, and typically also `nvidia-cudnn-cu12==9.20.0.48`, "
            "`nvidia-cusparselt-cu12==0.7.1`, `nvidia-nvshmem-cu12==3.4.5`, `cuda-toolkit==12.8.1`, "
            "and `cuda-bindings>=12.9.4,<13`. Attach the Kaggle dataset "
            "`masanorisuda/pytorch-runtime-cu128` and rerun."
        ) from exc
    raise

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Helper functions ---
if WHEEL_DIR:
    print(f"Deps wheel directory: {WHEEL_DIR}")
    whl_files = glob_mod.glob(os.path.join(WHEEL_DIR, "*.whl"))
    print(f"  Found {len(whl_files)} wheel files")
    for w in sorted(whl_files):
        print(f"    {os.path.basename(w)}")

def install_from_local_or_pip(module_name, pip_name=None, wheel_pattern=None):
    """Use a local wheel in offline mode; only hit pip if explicitly allowed."""
    try:
        import_module(module_name)
        print(f"  {module_name}: already installed")
        return
    except ImportError:
        pass

    if wheel_pattern:
        whl = _find_wheel(WHEEL_DIR, wheel_pattern)
        if whl and _wheel_seems_compatible(whl):
            whl = _fix_wheel_filename(whl)
            print(f"  {module_name}: installing from local wheel {os.path.basename(whl)}")
            _pip_install_local(whl, no_deps=True)
            import_module(module_name)
            return

    if ALLOW_ONLINE_INSTALL:
        print(f"  {module_name}: installing from pip")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module_name])
        import_module(module_name)
        return

    _raise_offline_dependency_error(module_name, wheel_pattern)

# --- Standard deps ---
for module_name in ["pandas", "transformers"]:
    import_module(module_name)

for module_name, pip_name, whl_pat in [
    ("peft", "peft", "peft-*.whl"),
    ("bitsandbytes", "bitsandbytes", "bitsandbytes-*.whl"),
    ("sentencepiece", "sentencepiece", "sentencepiece-*.whl"),
    ("safetensors", "safetensors", "safetensors-*.whl"),
]:
    install_from_local_or_pip(module_name, pip_name, whl_pat)

def _parse_torch_wheel_version(path):
    match = _re.search(r"torch(\d+\.\d+)", os.path.basename(path))
    if not match:
        return (0, 0)
    return tuple(int(part) for part in match.group(1).split("."))

def _find_best_local_mamba_wheel(package_name, cuda_major, torch_ver, py_ver, abi):
    exact_pattern = f"{package_name}-*+cu{cuda_major}torch{torch_ver}cxx11abi{abi}-cp{py_ver}-*.whl"
    exact = _find_wheel(WHEEL_DIR, exact_pattern)
    if exact:
        return exact, False

    fallback_pattern = f"{package_name}-*+cu{cuda_major}torch*cxx11abi{abi}-cp{py_ver}-*.whl"
    candidates = _find_wheels(WHEEL_DIR, fallback_pattern)
    if not candidates:
        return None, False

    candidates = sorted(candidates, key=_parse_torch_wheel_version, reverse=True)
    return candidates[0], True

# --- mamba-ssm / causal-conv1d ---
def _install_mamba_from_local_wheels():
    """Install mamba-ssm and causal-conv1d from local wheel files."""
    if not WHEEL_DIR:
        _raise_offline_dependency_error("mamba_ssm", extra_message="No local dependency dataset directory was found.")

    cuda_ver = torch.version.cuda
    if not cuda_ver:
        raise RuntimeError("torch.version.cuda is empty after setup. The GPU runtime is not usable.")

    cuda_major = int(cuda_ver.split(".")[0])
    torch_ver = ".".join(torch.__version__.split(".")[:2])
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    try:
        abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    except AttributeError:
        abi = "TRUE"

    print(f"Environment: CUDA {cuda_ver}, PyTorch {torch_ver}, Python {py_ver}, ABI={abi}")

    cc_whl, cc_fallback = _find_best_local_mamba_wheel("causal_conv1d", cuda_major, torch_ver, py_ver, abi)
    mamba_whl, mamba_fallback = _find_best_local_mamba_wheel("mamba_ssm", cuda_major, torch_ver, py_ver, abi)

    # Fallback: try generic wheel names (e.g. cross-compiled sm_120 wheels without version tags)
    if not cc_whl:
        generic = _find_wheel(WHEEL_DIR, f"causal_conv1d-*-cp{py_ver}-*.whl")
        if generic and _wheel_seems_compatible(generic):
            cc_whl, cc_fallback = generic, True
    if not mamba_whl:
        generic = _find_wheel(WHEEL_DIR, f"mamba_ssm-*-cp{py_ver}-*.whl")
        if generic and _wheel_seems_compatible(generic):
            mamba_whl, mamba_fallback = generic, True

    if not cc_whl:
        _raise_offline_dependency_error(
            "causal_conv1d",
            f"causal_conv1d-*-cp{py_ver}-*.whl",
            extra_message=f"Current runtime is torch {torch.__version__} on Python {py_ver}."
        )
    if not mamba_whl:
        _raise_offline_dependency_error(
            "mamba_ssm",
            f"mamba_ssm-*-cp{py_ver}-*.whl",
            extra_message=f"Current runtime is torch {torch.__version__} on Python {py_ver}."
        )

    if cc_fallback:
        print(
            f"  WARNING: No exact causal-conv1d wheel for torch {torch_ver}; "
            f"trying {os.path.basename(cc_whl)}"
        )
    if mamba_fallback:
        print(
            f"  WARNING: No exact mamba-ssm wheel for torch {torch_ver}; "
            f"trying {os.path.basename(mamba_whl)}"
        )

    cc_whl = _fix_wheel_filename(cc_whl)
    mamba_whl = _fix_wheel_filename(mamba_whl)

    print(f"causal-conv1d wheel: {os.path.basename(cc_whl)}")
    print(f"mamba-ssm wheel:     {os.path.basename(mamba_whl)}")

    _pip_install_local(cc_whl, force_reinstall=True, no_deps=True)
    _pip_install_local(mamba_whl, force_reinstall=True, no_deps=True)

# On Blackwell, always install from local wheels to override utility script versions
# (utility script versions lack sm_120 CUDA kernels)
gpu_name_pre, compute_cap_pre = _query_gpu_info()
_is_blackwell = False
if compute_cap_pre:
    try:
        _is_blackwell = int(compute_cap_pre.split(".")[0]) >= 12
    except Exception:
        pass

if _is_blackwell:
    print("Blackwell GPU: force-installing mamba_ssm/causal_conv1d from local wheels (overriding utility script)")
    _install_mamba_from_local_wheels()
else:
    try:
        import_module("mamba_ssm")
        print("mamba_ssm already installed")
    except ImportError:
        _install_mamba_from_local_wheels()

# Ensure our wheel install takes precedence over utility script paths
if RUNTIME_SITE_DIR in sys.path:
    sys.path.remove(RUNTIME_SITE_DIR)
sys.path.insert(0, RUNTIME_SITE_DIR)
# Reload mamba_ssm and causal_conv1d from our wheels
for _mod_name in list(sys.modules.keys()):
    if _mod_name.startswith(("mamba_ssm", "causal_conv1d", "selective_scan_cuda")):
        del sys.modules[_mod_name]
import_module("mamba_ssm")
import_module("causal_conv1d")

try:
    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        "Local mamba_ssm wheels could not be imported with the current runtime. "
        f"Current torch={torch.__version__}, cuda={torch.version.cuda}, python=cp{sys.version_info.major}{sys.version_info.minor}. "
        "For RTX PRO 6000 Blackwell, attach wheels built for this exact runtime "
        "(for example torch 2.12 / cu128 / cp312 if you upgraded PyTorch)."
    ) from exc

print("Dependencies ready (mamba_ssm verified)")


In [ ]:
# ============================================================
# 1. Config
# ============================================================
import json
import os
import random
import re
import zipfile
from pathlib import Path

import numpy as np

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
DEFAULT_MODEL_DIR = "/kaggle/input/nemotron-3-nano-30b-a3b-bf16"
MODEL_NAME = os.environ.get("NEMOTRON_MODEL_NAME", DEFAULT_MODEL_DIR)

# --- Locate files in /kaggle/input ---
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print("Contents of /kaggle/input:")
    for p in sorted(kaggle_input.iterdir()):
        print(f"  {p.name}/")

def _find_csv(name):
    """Search /kaggle/input for the competition CSV, with fallback to cwd."""
    if kaggle_input.exists():
        candidate = kaggle_input / "competitions" / "nvidia-nemotron-model-reasoning-challenge" / name
        if candidate.exists():
            return candidate
        candidate = kaggle_input / "nvidia-nemotron-model-reasoning-challenge" / name
        if candidate.exists():
            return candidate
        for p in sorted(kaggle_input.rglob(name)):
            return p
    cwd = Path.cwd() / name
    if cwd.exists():
        return cwd
    raise FileNotFoundError(f"Cannot find {name}")

TRAIN_CSV = _find_csv("train.csv")
TEST_CSV = _find_csv("test.csv")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
JSONL_DIR = WORK_DIR / "sft_data"
OUTPUT_DIR = WORK_DIR / "nemotron_lora_adapter"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
SUBMISSION_PATH = WORK_DIR / "submission.zip"

def _env_flag(name, default=False):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}

def _env_optional_int(name, default=None):
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return default
    if raw.strip().lower() == "none":
        return None
    return int(raw)

FAST_ITER = _env_flag("FAST_ITER", True)  # Set True for quick iteration, False for full run

# Unified defaults (FAST_ITER controls quick vs full)
_DEFAULTS = {
    "MAX_TRAIN_SAMPLES": 3000 if FAST_ITER else None,
    "NUM_EPOCHS": 1 if FAST_ITER else 2,
    "RUN_EVAL": True,
    "VALID_FRAC": 0.02 if FAST_ITER else 0.05,
}
if FAST_ITER:
    print(f"FAST_ITER=True: applying quick defaults {_DEFAULTS}")

MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "1024"))
VALID_FRAC = float(os.environ.get("VALID_FRAC", _DEFAULTS["VALID_FRAC"]))
NUM_EPOCHS = int(os.environ.get("NUM_EPOCHS", _DEFAULTS["NUM_EPOCHS"]))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "8"))
GRAD_ACCUM = int(os.environ.get("GRAD_ACCUM", "4"))
LEARNING_RATE = float(os.environ.get("LEARNING_RATE", "1.5e-4"))
LORA_R = int(os.environ.get("LORA_R", "32"))
LORA_ALPHA = int(os.environ.get("LORA_ALPHA", "64"))
LORA_DROPOUT = float(os.environ.get("LORA_DROPOUT", "0.05"))
MAX_TRAIN_SAMPLES = _env_optional_int("MAX_TRAIN_SAMPLES", _DEFAULTS["MAX_TRAIN_SAMPLES"])
RUN_EVAL = _env_flag("RUN_EVAL", _DEFAULTS["RUN_EVAL"])
SAVE_CHECKPOINTS = _env_flag("SAVE_CHECKPOINTS", False)
USE_GRADIENT_CHECKPOINTING = _env_flag("USE_GRADIENT_CHECKPOINTING", True)
TRAIN_LOGGING_STEPS = int(os.environ.get("TRAIN_LOGGING_STEPS", "1"))

SYSTEM_PROMPT = (
    "You are a precise puzzle solver. Given input-output examples showing a hidden rule, "
    "discover the rule and apply it to the new input. "
    "Output only the final answer inside \\boxed{}. Do not explain your reasoning."
)
INFER_SYSTEM_PROMPT = SYSTEM_PROMPT
SANITY_MAX_NEW_TOKENS = int(os.environ.get("SANITY_MAX_NEW_TOKENS", "64"))

# Hard / medium / easy puzzle types (by zero-shot difficulty)
HARD_TYPES = ["Bit Manipulation", "Text Encryption", "Equation Transformation"]
MEDIUM_TYPES = ["Gravitational Constant", "Unit Conversion"]
EASY_TYPES = ["Number Base Conversion"]

assert LORA_R <= 32
random.seed(SEED)
np.random.seed(SEED)
JSONL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME:  {MODEL_NAME}")
print(f"TRAIN_CSV:   {TRAIN_CSV}")
print(f"TEST_CSV:    {TEST_CSV}")
print(f"WORK_DIR:    {WORK_DIR}")
print(f"MAX_SEQ_LEN: {MAX_SEQ_LEN}")

In [ ]:
# ============================================================
# 2. Load CSVs, classify puzzles, balance, and build SFT JSONL
# ============================================================
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

assert list(train_df.columns) == ["id", "prompt", "answer"]
assert list(test_df.columns) == ["id", "prompt"]

if MAX_TRAIN_SAMPLES is not None:
    n = min(MAX_TRAIN_SAMPLES, len(train_df))
    train_df = train_df.sample(n=n, random_state=SEED).reset_index(drop=True)

# --- Puzzle type classifier ---
def classify_puzzle(prompt):
    p = prompt.lower()
    if re.search(r'numeral system|base[- ]?\d|number.*convert|radix|secret number', p):
        return 'Number Base Conversion'
    elif re.search(r'gravit|gravity|falling|free.?fall|acceleration due to', p):
        return 'Gravitational Constant'
    elif re.search(r'transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation', p):
        return 'Equation Transformation'
    elif re.search(r'encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text', p):
        return 'Text Encryption'
    elif re.search(r'bit.?manipul|binary|8.?bit|bitwise|bit.*transform', p):
        return 'Bit Manipulation'
    elif re.search(r'unit.?conver|measurement|becomes.*\d|secret.*conver.*measur', p):
        return 'Unit Conversion'
    return 'Unknown'

train_df['puzzle_type'] = train_df['prompt'].apply(classify_puzzle)
print("Puzzle type distribution:")
print(train_df['puzzle_type'].value_counts().to_string())



# --- Train/valid split FIRST (by unique ID, before any oversampling) ---
valid_size = max(1, int(round(len(train_df) * VALID_FRAC)))
valid_size = min(valid_size, len(train_df) - 1)
valid_df = train_df.sample(n=valid_size, random_state=SEED)
train_base_df = train_df.drop(valid_df.index).reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

# Verify no ID leakage
train_ids = set(train_base_df['id'].tolist())
valid_ids = set(valid_df['id'].tolist())
overlap = train_ids & valid_ids
assert len(overlap) == 0, f"ID leakage: {len(overlap)} IDs in both train and valid"
print(f"ID leakage check: 0 overlap (train={len(train_ids)}, valid={len(valid_ids)})")

# --- Per-type oversampling on TRAIN ONLY ---
# Bit Manipulation and Gravitational Constant boosted to 5x (weakest types)
OVERSAMPLE_RATES = {
    'Bit Manipulation':       5,
    'Text Encryption':        3,
    'Equation Transformation': 3,
    'Gravitational Constant': 5,
    'Unit Conversion':        2,
    'Number Base Conversion': 1,
}

_dfs = []
for _ptype, _rate in OVERSAMPLE_RATES.items():
    _df_t = train_base_df[train_base_df['puzzle_type'] == _ptype]
    if len(_df_t) > 0:
        _dfs.append(_df_t.sample(n=len(_df_t) * _rate, replace=True, random_state=SEED))
_unknown_df = train_base_df[~train_base_df['puzzle_type'].isin(OVERSAMPLE_RATES.keys())]
if len(_unknown_df) > 0:
    _dfs.append(_unknown_df)

if not _dfs:
    train_split_df = train_base_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
else:
    train_split_df = pd.concat(_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)

if MAX_TRAIN_SAMPLES is not None and len(train_split_df) > MAX_TRAIN_SAMPLES:
    train_split_df = train_split_df.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"Capped train rows to: {len(train_split_df)}")

print(f"\nOriginal: {len(train_df)} rows")
print(f"Train (after oversampling): {len(train_split_df)} rows")
print(f"Valid (clean, no overlap): {len(valid_df)} rows")
print(train_split_df['puzzle_type'].value_counts().to_string())



def to_record(row):
    answer = str(row['answer'])
    # Use CoT response if available (Gravity / Unit Conversion), else plain boxed
    cot = row.get('cot_response', None)
    if cot and str(cot) not in ('', 'nan', 'None'):
        assistant_response = str(cot)
    else:
        assistant_response = f"\\boxed{{{answer}}}"
    return {
        "id": row['id'],
        "prompt": row['prompt'],
        "answer": answer,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row['prompt']},
            {"role": "assistant", "content": assistant_response},
        ],
    }

# --- CoT response generation (Gravitational Constant / Unit Conversion) ---
def _make_gravity_cot(prompt):
    pairs = re.findall(r't\s*=\s*([\d.]+)s,\s*distance\s*=\s*([\d.]+)\s*m', prompt)
    if not pairs:
        return None
    qm = re.search(r'for t\s*=\s*([\d.]+)s', prompt.split('Now,')[-1])
    if not qm:
        return None
    qt = float(qm.group(1))
    lines, gs = [], []
    for ts, ds in pairs:
        t, d = float(ts), float(ds)
        if t == 0:
            continue
        g = 2 * d / t ** 2
        gs.append(g)
        lines.append(f"g = 2 × {d} / {t}² = {g:.4f}")
    if not gs:
        return None
    gm = np.mean(gs)
    ans = round(0.5 * gm * qt ** 2, 2)
    return "\n".join(lines) + f"\ng_mean = {gm:.4f}\nd = 0.5 × {gm:.4f} × {qt}² = {ans}\n\\boxed{{{ans}}}"

def _make_unit_cot(prompt):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    if not pairs:
        return None
    qm = re.search(r'convert the following measurement:\s*([\d.]+)', prompt)
    if not qm:
        qm = re.search(r'([\d.]+)\s*m\s*$', prompt.strip())
    if not qm:
        return None
    qv = float(qm.group(1))
    lines, ratios = [], []
    for i_str, o_str in pairs:
        inp, out = float(i_str), float(o_str)
        if inp == 0:
            continue
        r = out / inp
        ratios.append(r)
        lines.append(f"ratio = {out} / {inp} = {r:.4f}")
    if not ratios:
        return None
    rm = np.mean(ratios)
    ans = round(qv * rm, 2)
    return "\n".join(lines) + f"\nratio_mean = {rm:.4f}\nanswer = {qv} × {rm:.4f} = {ans}\n\\boxed{{{ans}}}"

def _assign_cot(row):
    pt = row.get('puzzle_type', '')
    if pt == 'Gravitational Constant':
        return _make_gravity_cot(row['prompt'])
    elif pt == 'Unit Conversion':
        return _make_unit_cot(row['prompt'])
    return None

train_split_df['cot_response'] = train_split_df.apply(_assign_cot, axis=1)
_n_cot = train_split_df['cot_response'].notna().sum()
print(f"CoT generated: {_n_cot}/{len(train_split_df)} rows (Gravity + Unit Conversion)")

train_jsonl = JSONL_DIR / "train_sft.jsonl"
valid_jsonl = JSONL_DIR / "valid_sft.jsonl"

with train_jsonl.open("w", encoding="utf-8") as f:
    for _, row in train_split_df.iterrows():
        f.write(json.dumps(to_record(row), ensure_ascii=False) + "\n")

with valid_jsonl.open("w", encoding="utf-8") as f:
    for _, row in valid_df.iterrows():
        f.write(json.dumps(to_record(row), ensure_ascii=False) + "\n")

print(f"\ntrain rows: {len(train_split_df):,}")
print(f"valid rows: {len(valid_df):,}")
print(f"test rows:  {len(test_df):,}")

# Preview
preview = to_record(train_split_df.iloc[0])
print(f"\nSample assistant response:\n{preview['messages'][2]['content'][:300]}")

In [ ]:
# ============================================================
# 3. Load tokenizer and prompt helpers
# ============================================================
import torch
from transformers import AutoTokenizer

def _find_model_dir(base_name):
    """Search /kaggle/input for the model directory containing config.json."""
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        # List all input directories for debugging
        print("Searching for model in /kaggle/input:")
        for p in sorted(kaggle_input.iterdir()):
            print(f"  {p.name}/")
        # Search for config.json in all subdirectories
        for config in sorted(kaggle_input.rglob("config.json")):
            model_dir = str(config.parent)
            print(f"  Found model config at: {model_dir}")
            return model_dir
    # Fallback to the configured name
    return base_name

model_path_candidate = Path(MODEL_NAME)
if model_path_candidate.exists():
    # Check if config.json is directly here or in a subdirectory
    if (model_path_candidate / "config.json").exists():
        MODEL_PATH = str(model_path_candidate)
    else:
        # Search for config.json in subdirectories (Kaggle Models structure)
        configs = list(model_path_candidate.rglob("config.json"))
        if configs:
            MODEL_PATH = str(configs[0].parent)
            print(f"Found model config in subdirectory: {MODEL_PATH}")
        else:
            MODEL_PATH = str(model_path_candidate)
    local_model = True
else:
    # Try to find model elsewhere in /kaggle/input
    MODEL_PATH = _find_model_dir(MODEL_NAME)
    if Path(MODEL_PATH).exists():
        local_model = True
    else:
        local_model = False
        if MODEL_NAME.startswith("/") or MODEL_NAME.startswith("."):
            raise FileNotFoundError(
                f"Base model not found: {MODEL_NAME}\n"
                "Attach the Nemotron base model via Kaggle Models, "
                "or set NEMOTRON_MODEL_NAME to a valid model path."
            )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=local_model,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

def build_messages(prompt, answer=None, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer})
    return messages

def render_messages(messages, add_generation_prompt=False):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    chunks = []
    for message in messages:
        chunks.append(f"{message['role'].upper()}: {message['content']}")
    if add_generation_prompt:
        chunks.append("ASSISTANT:")
    return "\n".join(chunks)

preview_row = train_split_df.iloc[0]
preview_response = to_record(preview_row)["messages"][2]["content"]
preview_text = render_messages(
    build_messages(
        preview_row["prompt"],
        preview_response,
    )
)
print(f"MODEL_PATH: {MODEL_PATH}")
print(preview_text[:800])
print(f"chat_template: {bool(getattr(tokenizer, 'chat_template', None))}")
print(f"compute_dtype: {compute_dtype}")

In [ ]:
# ============================================================
# 4. Build tokenized train/valid datasets
# ============================================================
from torch.utils.data import Dataset

def preprocess(example):
    messages = example["messages"]
    full_text = render_messages(messages, add_generation_prompt=False)
    prompt_text = render_messages(messages[:-1], add_generation_prompt=True)

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    labels = input_ids.copy()
    prompt_len = min(len(prompt_tokens["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_records = [json.loads(line) for line in train_jsonl.read_text(encoding="utf-8").splitlines()]
valid_records = [json.loads(line) for line in valid_jsonl.read_text(encoding="utf-8").splitlines()]

train_features = [preprocess(record) for record in train_records]
valid_features = [preprocess(record) for record in valid_records]

class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

train_dataset = ListDataset(train_features)
valid_dataset = ListDataset(valid_features)

def data_collator(features):
    model_inputs = [
        {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
        for f in features
    ]
    batch = tokenizer.pad(model_inputs, padding=True, return_tensors="pt")
    max_len = batch["input_ids"].shape[1]
    batch["labels"] = torch.tensor(
        [f["labels"] + [-100] * (max_len - len(f["labels"])) for f in features],
        dtype=torch.long,
    )
    return batch

print(f"train examples: {len(train_dataset):,}")
print(f"valid examples: {len(valid_dataset):,}")
print(train_dataset[0].keys())


In [ ]:
# ============================================================
# 5. Load base model and attach LoRA
# ============================================================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Detect GPU VRAM
gpu_mem_gb = 0
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}, VRAM: {gpu_mem_gb:.1f} GB")


def _disable_blackwell_unsafe_mamba_fast_path(model):
    import sys

    if not torch.cuda.is_available():
        return

    if torch.cuda.get_device_capability(0) < (12, 0):
        return

    patched_modules = []
    module_names = set()
    for obj in (
        model,
        getattr(model, "model", None),
        getattr(model, "backbone", None),
        getattr(model, "base_model", None),
        getattr(getattr(model, "model", None), "backbone", None),
    ):
        mod_name = getattr(getattr(obj, "__class__", None), "__module__", None)
        if mod_name:
            module_names.add(mod_name)

    for mod_name in sorted(module_names):
        module = sys.modules.get(mod_name)
        if module is None:
            continue
        changed = []
        for attr in dir(module):
            if "fast_path" not in attr.lower():
                continue
            try:
                value = getattr(module, attr)
            except Exception:
                continue
            if isinstance(value, bool):
                setattr(module, attr, False)
                changed.append(attr)
        if changed:
            patched_modules.append((mod_name, changed))

    instance_attrs = ("is_fast_path_available", "use_fast_path", "_use_fast_path")
    patched_instances = 0
    for submodule in model.modules():
        for attr in instance_attrs:
            if not hasattr(submodule, attr):
                continue
            try:
                value = getattr(submodule, attr)
            except Exception:
                continue
            if isinstance(value, bool):
                try:
                    setattr(submodule, attr, False)
                    patched_instances += 1
                except Exception:
                    pass

    if patched_modules or patched_instances:
        print("Disabled Blackwell-unsafe Mamba fast path")
        for mod_name, attrs in patched_modules:
            print(f"  {mod_name}: {', '.join(sorted(set(attrs)))}")
        if patched_instances:
            print(f"  patched instance flags: {patched_instances}")
    else:
        print("Blackwell detected, but no explicit Mamba fast-path flags were found.")


def _patch_nemotron_moe_dtype_mismatch(model):
    import sys

    patched = []
    module_names = set()
    for obj in (
        model,
        getattr(model, "model", None),
        getattr(model, "backbone", None),
        getattr(model, "base_model", None),
        getattr(getattr(model, "model", None), "backbone", None),
    ):
        mod_name = getattr(getattr(obj, "__class__", None), "__module__", None)
        if mod_name:
            module_names.add(mod_name)

    for mod_name in sorted(module_names):
        module = sys.modules.get(mod_name)
        if module is None:
            continue
        moe_cls = getattr(module, "NemotronHMOE", None)
        if moe_cls is None:
            continue
        if getattr(moe_cls, "_dtype_fix_patched", False):
            patched.append(mod_name)
            continue

        def patched_moe(self, hidden_states, topk_indices, topk_weights):
            final_hidden_states = torch.zeros_like(hidden_states, dtype=hidden_states.dtype)
            expert_mask = torch.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
            expert_mask = expert_mask.permute(2, 0, 1)

            for expert_idx in range(len(self.experts)):
                expert = self.experts[expert_idx]
                mask = expert_mask[expert_idx]
                token_indices, weight_indices = torch.where(mask)

                if token_indices.numel() > 0:
                    expert_weights = topk_weights[token_indices, weight_indices].to(hidden_states.dtype)
                    expert_input = hidden_states[token_indices]
                    expert_output = expert(expert_input).to(hidden_states.dtype)
                    weighted_output = expert_output * expert_weights.unsqueeze(-1)
                    final_hidden_states.index_add_(0, token_indices, weighted_output)
                else:
                    dummy_input = hidden_states[:1].detach().clone().zero_().to(hidden_states.dtype)
                    dummy_out = expert(dummy_input).to(hidden_states.dtype)
                    final_hidden_states = final_hidden_states + dummy_out

            return final_hidden_states

        def patched_forward(self, hidden_states):
            residuals = hidden_states
            orig_shape = hidden_states.shape
            topk_indices, topk_weights = self.gate(hidden_states)
            hidden_states = hidden_states.view(-1, hidden_states.shape[-1])
            hidden_states = self.moe(hidden_states, topk_indices, topk_weights).view(*orig_shape)
            shared_output = self.shared_experts(residuals).to(hidden_states.dtype)
            hidden_states = hidden_states + shared_output
            return hidden_states

        moe_cls.moe = patched_moe
        moe_cls.forward = patched_forward
        moe_cls._dtype_fix_patched = True
        patched.append(mod_name)

    if patched:
        print("Patched Nemotron MoE dtype handling:")
        for mod_name in patched:
            print(f"  {mod_name}.NemotronHMOE")


def _maybe_disable_blackwell_fast_path(model):
    """Only disable Mamba fast_path if CUDA extensions don't support sm_120."""
    if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) < (12, 0):
        return
    # Test if selective_scan_cuda actually works on this GPU
    try:
        import selective_scan_cuda  # noqa: F401
        print("Mamba CUDA extensions loaded successfully on Blackwell — fast_path ENABLED")
    except (ImportError, OSError) as exc:
        print(f"Mamba CUDA extensions not compatible with Blackwell ({exc})")
        print("Disabling Mamba fast_path for safety...")
        _disable_blackwell_unsafe_mamba_fast_path(model)


load_kwargs = {"trust_remote_code": True}
if local_model:
    load_kwargs["local_files_only"] = True

if gpu_mem_gb >= 48:
    print(f"Large VRAM ({gpu_mem_gb:.0f}GB), loading in bf16 (no quantization)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=compute_dtype,
        device_map="auto",
        **load_kwargs,
    )
    _maybe_disable_blackwell_fast_path(model)
    _patch_nemotron_moe_dtype_mismatch(model)
    model.config.use_cache = False
    model.enable_input_require_grads()
else:
    print(f"Limited VRAM ({gpu_mem_gb:.0f}GB), loading with 4bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    if 0 < gpu_mem_gb < 24:
        load_kwargs["max_memory"] = {0: f"{int(gpu_mem_gb * 0.85)}GiB", "cpu": "24GiB"}
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        **load_kwargs,
    )
    _maybe_disable_blackwell_fast_path(model)
    _patch_nemotron_moe_dtype_mismatch(model)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Compile model for faster training
if hasattr(torch, "compile"):
    try:
        model = torch.compile(model)
        print("torch.compile applied successfully")
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")


In [ ]:
# ============================================================
# 6. Train
# ============================================================
from inspect import signature

from transformers import Trainer, TrainerCallback, TrainingArguments


TRAIN_LOG_PATH = WORK_DIR / "train_log.csv"

class HeartbeatCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self._log_file = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._log_file = open(TRAIN_LOG_PATH, "w", encoding="utf-8")
        self._log_file.write("step,epoch,loss,learning_rate\n")
        self._log_file.flush()

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        parts = [f"step={state.global_step}"]
        epoch = logs.get("epoch", "")
        loss = logs.get("loss", "")
        lr = logs.get("learning_rate", "")
        if epoch:
            parts.append(f"epoch={epoch:.4f}")
        if loss != "":
            parts.append(f"loss={loss:.6f}")
        if lr != "":
            parts.append(f"lr={lr:.3e}")
        print("train:", ", ".join(parts), flush=True)
        if self._log_file and loss != "":
            self._log_file.write(f"{state.global_step},{epoch},{loss},{lr}\n")
            self._log_file.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self._log_file:
            self._log_file.close()
            self._log_file = None
            print(f"Training log saved to {TRAIN_LOG_PATH}")


if not USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

save_strategy = "epoch" if SAVE_CHECKPOINTS else "no"
eval_strategy_value = "epoch" if RUN_EVAL else "no"

training_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=TRAIN_LOGGING_STEPS,
    save_strategy=save_strategy,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    optim="paged_adamw_8bit" if gpu_mem_gb < 48 else "adamw_torch_fused",
    dataloader_num_workers=2,
    bf16=supports_bf16,
    fp16=not supports_bf16,
)
if "eval_strategy" in signature(TrainingArguments.__init__).parameters:
    training_kwargs["eval_strategy"] = eval_strategy_value
else:
    training_kwargs["evaluation_strategy"] = eval_strategy_value

supported_training_kwargs = {
    key: value for key, value in training_kwargs.items()
    if key in signature(TrainingArguments.__init__).parameters
}
skipped_training_kwargs = sorted(set(training_kwargs) - set(supported_training_kwargs))
if skipped_training_kwargs:
    print(f"Skipping unsupported TrainingArguments kwargs: {skipped_training_kwargs}")

print(
    f"Fast-train config: max_seq_len={MAX_SEQ_LEN}, max_train_samples={MAX_TRAIN_SAMPLES}, "
    f"run_eval={RUN_EVAL}, gradient_checkpointing={USE_GRADIENT_CHECKPOINTING}, "
    f"logging_steps={TRAIN_LOGGING_STEPS}, save_checkpoints={SAVE_CHECKPOINTS}"
)

training_args = TrainingArguments(**supported_training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset if RUN_EVAL else None,
    data_collator=data_collator,
    callbacks=[HeartbeatCallback()],
)

train_metrics = trainer.train().metrics
print(train_metrics)

# Skip trainer.evaluate() — custom per-type eval runs in the next cell
print("Trainer evaluation skipped (custom eval in next cell)")


In [ ]:
# ============================================================
# 7. Save adapter and run a quick inference sanity check
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

# Create submission.zip BEFORE eval (guard against timeout)
required_files = ["adapter_config.json"]
weight_candidates = ["adapter_model.safetensors", "adapter_model.bin"]
missing = [name for name in required_files if not (OUTPUT_DIR / name).exists()]
if not any((OUTPUT_DIR / name).exists() for name in weight_candidates):
    missing.append("adapter_model.safetensors or adapter_model.bin")
if missing:
    raise FileNotFoundError(f"Missing adapter files in {OUTPUT_DIR}: {missing}")
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)
            print(f"added: {path.name}")
print(f"submission (pre-eval): {SUBMISSION_PATH} ({SUBMISSION_PATH.stat().st_size / 1e6:.2f} MB)")

def _is_placeholder_answer(value):
    normalized = value.strip().lower().strip("{}<>")
    placeholders = {
        "answer",
        "final answer",
        "actual answer",
        "your answer",
        "the answer",
        "actual_solved_value",
        "actual solution",
        "solution",
    }
    return normalized in placeholders or "placeholder" in normalized

def extract_boxed(text):
    # Extract answer from \boxed{...}. Minimal fallback to avoid false matches.
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    for match in reversed(matches):
        candidate = match.strip()
        if candidate and not _is_placeholder_answer(candidate):
            return candidate

    # Fallback: look for "Final:" prefix
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        if line.lower().startswith("final:"):
            return line.split(":", 1)[1].strip()

    # Last resort: return last non-empty line (but warn)
    if lines:
        print(f"  WARNING: no \\boxed{{}} found, using last line as answer")
        return lines[-1]
    return text.strip()

def _eos_token_ids():
    eos_ids = []
    for token in [getattr(tokenizer, "eos_token", None), "<|im_end|>", "<|eot_id|>"]:
        if token is None:
            continue
        token_id = tokenizer.convert_tokens_to_ids(token) if isinstance(token, str) else token
        if isinstance(token_id, int) and token_id >= 0 and token_id != getattr(tokenizer, "unk_token_id", None):
            eos_ids.append(token_id)
    if not eos_ids and getattr(tokenizer, "eos_token_id", None) is not None:
        eos_ids.append(tokenizer.eos_token_id)
    return sorted(set(eos_ids))

def _generate_raw(prompt, max_new_tokens, system_prompt=None):
    messages = build_messages(prompt, system_prompt=system_prompt or INFER_SYSTEM_PROMPT)
    prompt_text = render_messages(messages, add_generation_prompt=True)
    device = trainer.model.get_input_embeddings().weight.device
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    trainer.model.eval()
    generation_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    eos_ids = _eos_token_ids()
    if eos_ids:
        generation_kwargs["eos_token_id"] = eos_ids

    with torch.no_grad():
        output_ids = trainer.model.generate(
            **inputs,
            **generation_kwargs,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Truncate after first complete \boxed{...}
    boxed_start = text.find("\\boxed{")
    if boxed_start >= 0:
        brace_depth = 0
        for i in range(boxed_start + len("\\boxed{"), len(text)):
            if text[i] == '{':
                brace_depth += 1
            elif text[i] == '}':
                if brace_depth == 0:
                    text = text[:i + 1]
                    break
                brace_depth -= 1
    return text

def generate_answer(prompt, max_new_tokens=SANITY_MAX_NEW_TOKENS):
    raw = _generate_raw(prompt, max_new_tokens=max_new_tokens)
    extracted = extract_boxed(raw)
    if extracted and not _is_placeholder_answer(extracted):
        return extracted, raw
    # Fallback: use first line of raw output
    answer = raw.strip().split("\n")[0].strip()
    return answer, raw

# --- Per-type evaluation on full valid set ---
print(f"\n=== Evaluation on {len(valid_df)} valid samples ===")
eval_results = []

for idx, row in valid_df.iterrows():
    prompt = row["prompt"]
    target = str(row["answer"]).strip()
    puzzle_type = row.get("puzzle_type", "Unknown")

    extracted, raw = generate_answer(prompt)
    extracted = str(extracted).strip()

    exact_match = (extracted == target)
    has_boxed = "\\boxed{" in raw
    # format_valid: output is clean boxed-only (no extra text outside boxed/think)
    _clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    format_valid = bool(re.fullmatch(r'\\boxed\{[^}]*\}', _clean))
    # reasoning_leakage: output has reasoning text (before or instead of boxed)
    _leakage_keywords = ('we need', 'let me', 'the rule', 'looking at', 'first,',
                         'step', 'therefore', 'so the', 'thus', 'notice')
    reasoning_leakage = (not format_valid) and any(kw in raw.lower() for kw in _leakage_keywords)
    eval_results.append({
        "puzzle_type": puzzle_type,
        "exact_match": exact_match,
        "has_boxed": has_boxed,
        "format_valid": format_valid,
        "reasoning_leakage": reasoning_leakage,
        "extracted": extracted,
        "target": target,
    })

# Summary
import collections
type_stats = collections.defaultdict(lambda: {"total": 0, "correct": 0, "boxed": 0, "fmt_valid": 0, "leakage": 0})
for r in eval_results:
    pt = r["puzzle_type"]
    type_stats[pt]["total"] += 1
    type_stats[pt]["correct"] += int(r["exact_match"])
    type_stats[pt]["boxed"] += int(r["has_boxed"])
    type_stats[pt]["fmt_valid"] += int(r["format_valid"])
    type_stats[pt]["leakage"] += int(r["reasoning_leakage"])

total_correct = sum(r["exact_match"] for r in eval_results)
total_boxed = sum(r["has_boxed"] for r in eval_results)
total_fmt = sum(r["format_valid"] for r in eval_results)
total_leak = sum(r["reasoning_leakage"] for r in eval_results)
total = len(eval_results)

print(f"\nOverall: {total_correct}/{total} correct ({100*total_correct/total:.1f}%), "
      f"{total_boxed}/{total} boxed ({100*total_boxed/total:.1f}%), "
      f"{total_fmt}/{total} fmt_valid ({100*total_fmt/total:.1f}%), "
      f"{total_leak}/{total} reasoning_leakage ({100*total_leak/total:.1f}%)")
print(f"\n{'Type':<30s} {'Correct':>10s} {'Boxed':>8s} {'FmtOK':>7s} {'Leak':>6s}")
print("-" * 68)
for pt in sorted(type_stats.keys()):
    s = type_stats[pt]
    n = s["total"]
    print(f"{pt:<30s} {s['correct']:>3d}/{n:<3d} ({100*s['correct']/n:5.1f}%)"
          f" {s['boxed']:>3d}/{n:<3d} ({100*s['boxed']/n:4.0f}%)"
          f" {s['fmt_valid']:>3d}/{n:<3d} ({100*s['fmt_valid']/n:4.0f}%)"
          f" {s['leakage']:>3d}/{n:<3d} ({100*s['leakage']/n:4.0f}%)")

# Show wrong examples
wrong = [r for r in eval_results if not r["exact_match"]]
if wrong:
    print(f"\n--- Sample errors (up to 5) ---")
    for r in wrong[:5]:
        print(f"  [{r['puzzle_type']}] expected=\"{r['target']}\" got=\"{r['extracted'][:80]}\"")

# Save eval results to JSON for download/analysis
eval_output = {
    "overall": {
        "total": total,
        "correct": total_correct,
        "accuracy": round(100 * total_correct / total, 2) if total else 0,
        "boxed_rate": round(100 * total_boxed / total, 2) if total else 0,
    },
    "by_type": {
        pt: {
            "total": s["total"],
            "correct": s["correct"],
            "accuracy": round(100 * s["correct"] / s["total"], 2) if s["total"] else 0,
            "boxed_rate": round(100 * s["boxed"] / s["total"], 2) if s["total"] else 0,
            "format_valid_rate": round(100 * s["fmt_valid"] / s["total"], 2) if s["total"] else 0,
            "reasoning_leakage_rate": round(100 * s["leakage"] / s["total"], 2) if s["total"] else 0,
        }
        for pt, s in sorted(type_stats.items())
    },
    "details": eval_results,
}
eval_json_path = WORK_DIR / "eval_results.json"
with open(eval_json_path, "w", encoding="utf-8") as f:
    json.dump(eval_output, f, ensure_ascii=False, indent=2)
print(f"\nEval results saved to {eval_json_path}")


In [ ]:
# ============================================================
# 8. Create submission.zip
# ============================================================
required_files = ["adapter_config.json"]
weight_candidates = ["adapter_model.safetensors", "adapter_model.bin"]

missing = [name for name in required_files if not (OUTPUT_DIR / name).exists()]
if not any((OUTPUT_DIR / name).exists() for name in weight_candidates):
    missing.append("adapter_model.safetensors or adapter_model.bin")
if missing:
    raise FileNotFoundError(f"Missing adapter files in {OUTPUT_DIR}: {missing}")

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)
            print(f"added: {path.name}")

print(f"submission: {SUBMISSION_PATH}")
print(f"size_mb: {SUBMISSION_PATH.stat().st_size / 1e6:.2f}")
